## Data Transition and Modeling Readiness

The assessment focuses on:

- Evaluating the dataset's suitability for demand forecasting.
- Identifying potential target variables and predictive features.
- Assessing temporal data quality and forecasting requirements.
- Identifying the information required for operational optimization.
- Establishing a transition from historical demand analysis to predictive and prescriptive analytics.

In [4]:
import pandas as pd
import numpy as np

data = pd.read_csv("hour.csv")

data["dteday"] = pd.to_datetime(data["dteday"])

data["datetime"] = (
    data["dteday"]
    + pd.to_timedelta(data["hr"], unit="h")
)

data = data.sort_values("datetime").reset_index(drop=True)

# ------------------------------------------------------------

print("\n 1. DATASET OVERVIEW")
print("=" * 50)

overview = pd.DataFrame({
    "Metric": [
        "Number of Records",
        "Number of Features",
        "Start Date",
        "End Date",
        "Number of Unique Days",
        "Number of Unique Hours",
        "Missing Values",
        "Duplicate Timestamps"
    ],
    "Value": [
        len(data),
        len(data.columns),
        data["datetime"].min(),
        data["datetime"].max(),
        data["dteday"].nunique(),
        data["hr"].nunique(),
        data.isnull().sum().sum(),
        data["datetime"].duplicated().sum()
    ]
})

display(overview)

# ------------------------------------------------------------

print("\n 2. FORECASTING TARGET ASSESSMENT")
print("=" * 100)

target_summary = data["cnt"].describe().to_frame().T

target_summary["Skewness"] = data["cnt"].skew()

target_summary["Zero Demand Percentage"] = (
    (data["cnt"] == 0).mean() * 100
)

target_summary["Coefficient of Variation"] = (
    data["cnt"].std() / data["cnt"].mean()
)

display(target_summary.round(3))

# ------------------------------------------------------------

print("\n 3. TEMPORAL DATA QUALITY")
print("=" * 50)

expected_timestamps = pd.date_range(
    start=data["datetime"].min(),
    end=data["datetime"].max(),
    freq="h"
)

missing_timestamps = expected_timestamps.difference(
    data["datetime"]
)

daily_completeness = (
    data.groupby("dteday")
    .size()
    .rename("Observed Hours")
    .to_frame()
)

daily_completeness["Missing Hours"] = (
    24 - daily_completeness["Observed Hours"]
)

print("Expected hourly timestamps:", len(expected_timestamps))
print("Missing timestamps:", len(missing_timestamps))
print(
    "Days with 24 observations:",
    (daily_completeness["Observed Hours"] == 24).sum()
)
print(
    "Days with incomplete observations:",
    (daily_completeness["Observed Hours"] < 24).sum()
)

display(
    daily_completeness
    .sort_values("Missing Hours", ascending=False)
    .head(10)
)

# ------------------------------------------------------------

print("\n 4. FORECASTING FEATURE READINESS")
print("=" * 50)

candidate_features = [
    "hr",
    "weekday",
    "workingday",
    "holiday",
    "season",
    "mnth",
    "weathersit",
    "temp",
    "atemp",
    "hum",
    "windspeed",
    "casual",
    "registered"
]

feature_assessment = pd.DataFrame({
    "Feature": candidate_features,
    "Available": [
        feature in data.columns
        for feature in candidate_features
    ],
    "Data Type": [
        str(data[feature].dtype)
        if feature in data.columns
        else "Not Available"
        for feature in candidate_features
    ]
})

display(feature_assessment)

# ------------------------------------------------------------

print("\n 5. FORECASTING GRANULARITY ASSESSMENT")
print("=" * 50)

hourly_target = data["cnt"]

daily_target = (
    data.groupby("dteday")["cnt"]
    .sum()
)

print("Hourly forecasting observations:", len(hourly_target))
print("Daily forecasting observations:", len(daily_target))

print("\nHourly Demand Statistics:")
display(hourly_target.describe().round(2))

print("\nDaily Demand Statistics:")
display(daily_target.describe().round(2))


 1. DATASET OVERVIEW


,Metric,Value
0,Number of Records,17379
1,Number of Features,18
2,Start Date,2011-01-01 00:00:00
3,End Date,2012-12-31 23:00:00
4,Number of Unique Days,731
5,Number of Unique Hours,24
6,Missing Values,0
7,Duplicate Timestamps,0



 2. FORECASTING TARGET ASSESSMENT


,count,mean,std,min,25%,50%,75%,max,Skewness,Zero Demand Percentage,Coefficient of Variation
cnt,17379.0,189.463,181.388,1.0,40.0,142.0,281.0,977.0,1.277,0.0,0.957



 3. TEMPORAL DATA QUALITY
Expected hourly timestamps: 17544
Missing timestamps: 165
Days with 24 observations: 655
Days with incomplete observations: 76


,Observed Hours,Missing Hours
dteday,,
2012-10-29,1,23
2011-01-27,8,16
2012-10-30,11,13
2011-01-18,12,12
2011-01-26,16,8
2011-08-28,17,7
2011-02-22,18,6
2011-08-27,18,6
2011-02-11,22,2


... (2 outputs left out)


 5. FORECASTING GRANULARITY ASSESSMENT
Hourly forecasting observations: 17379
Daily forecasting observations: 731

Hourly Demand Statistics:


count    17379.00
mean       189.46
std        181.39
min          1.00
25%         40.00
50%        142.00
75%        281.00
max        977.00
Name: cnt, dtype: float64


Daily Demand Statistics:


count     731.00
mean     4504.35
std      1937.21
min        22.00
25%      3152.00
50%      4548.00
75%      5956.00
max      8714.00
Name: cnt, dtype: float64

## Overall Exploratory Data Analysis Findings

**Demand Characteristics**

The hourly demand distribution is positively skewed, with a mean of 189.46 and a median of 142 rentals. The skewness value of 1.277 indicates that high-demand observations influence the overall distribution. Furthermore, the coefficient of variation of 0.957 highlights substantial demand variability, suggesting that forecasting models should capture fluctuations rather than relying solely on average demand patterns.

**Temporal Data Quality**

Although the dataset contains no missing values within the observed records, 165 hourly timestamps are missing across the two-year observation period. Additionally, 76 days have incomplete hourly observations. These gaps should be considered during forecasting data preparation to avoid introducing artificial demand values or misleading temporal patterns.

**Forecasting Readiness**

The dataset provides several potentially useful explanatory variables, including time-related features, weather conditions, temperature, humidity, and wind speed. These variables offer a foundation for developing an hourly demand forecasting model.

The hourly forecasting granularity is particularly relevant because it preserves short-term demand fluctuations and supports more detailed operational planning.

**Optimization Readiness**

Historical rental demand provides an initial understanding of resource requirements. However, an optimization model requires additional operational parameters, such as fleet availability, station capacity, travel times, operational costs, and service-level constraints.

These parameters will need to be defined through additional data sources or clearly stated operational assumptions.

**Overall Conclusion**

The exploratory analysis confirms that the dataset is suitable as a starting point for demand forecasting. The next stage will focus on preparing a forecasting-ready dataset, engineering relevant temporal features, preventing target leakage, and preserving chronological order for model development and evaluation.

The resulting demand forecasts will subsequently serve as inputs for the capacity planning and resource allocation optimization stage.